# Homework 1 — Taming the Imbalanced Dataset

**Course:** Introduction to Machine Learning — TeknoLAB  
**Dataset:** German Credit Risk  
**Goal:** Improve recall and F1-score for the Risk class (class 1) using three imbalance-handling techniques with logistic regression.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

Target encoding: **0 = Safe**, **1 = Risk**

In [ ]:
df = pd.read_csv('data/german_credit.csv')

print('Class distribution (original):')
print(df['class'].value_counts())

# Encode: 'good' -> 0 (Safe), 'bad' -> 1 (Risk)
df['target'] = (df['class'] == 'bad').astype(int)

print('\nAfter target encoding (0=Safe, 1=Risk):')
print(df['target'].value_counts())
print(f'\nShape: {df.shape}')

Class distribution (original):
class
good    700
bad     300
Name: count, dtype: int64

After target encoding (0=Safe, 1=Risk):
target
0    700
1    300
Name: count, dtype: int64

Shape: (1000, 22)


The dataset is imbalanced: 70% Safe vs 30% Risk. This means a naive model that always predicts Safe achieves 70% accuracy — but completely misses all risky applicants.

## 3. Preprocessing

In [ ]:
X = df.drop(columns=['class', 'target'])
y = df['target']

numerical_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical features ({len(numerical_features)}): {numerical_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

# Preprocessor: scale numerical, one-hot encode categorical
# Fit is applied only on training data via Pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

Numerical features (7): ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
Categorical features (13): ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'telephone', 'foreign_worker']


## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train size: {X_train.shape[0]}   Test size: {X_test.shape[0]}')
print('\nTrain distribution:\n', y_train.value_counts())
print('\nTest distribution:\n', y_test.value_counts())

Train size: 800   Test size: 200

Train distribution:
 0    560
 1    240
Name: target, dtype: int64

Test distribution:
 0    140
 1     60
Name: target, dtype: int64


Stratified split preserves the 70/30 class ratio in both train and test sets. **The test set is kept untouched throughout development.**

## 5. Helper Function

In [ ]:
def evaluate(y_true, y_pred, label=''):
    """Print and return metrics for class 1."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    if label:
        print(f'\n--- {label} ---')
        print(classification_report(y_true, y_pred, target_names=['Safe (0)', 'Risk (1)']))
    return {'accuracy': round(acc, 4), 'precision': round(prec, 4),
            'recall': round(rec, 4), 'f1': round(f1, 4)}

## 6. Baseline Model — Logistic Regression

In [ ]:
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)

metrics = {}
metrics['Baseline'] = evaluate(y_test, y_pred_baseline, 'Baseline: LogisticRegression')


--- Baseline: LogisticRegression ---
              precision    recall  f1-score   support

    Safe (0)       0.76      0.90      0.83       140
    Risk (1)       0.60      0.35      0.44        60

    accuracy                           0.73       200
   macro avg       0.68      0.62      0.63       200
weighted avg       0.71      0.73      0.71       200



The baseline achieves 73.5% accuracy but recall for the Risk class is only **0.35** — it misses 65% of risky applicants.

## 7. Technique 1 — Cost-Sensitive Learning (class_weight='balanced')

**Theory:** Class weights penalize misclassifying minority-class examples more heavily during training. The `'balanced'` option automatically sets weights inversely proportional to class frequencies: `n_samples / (n_classes * np.bincount(y))`.

In [ ]:
cw_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

cw_pipeline.fit(X_train, y_train)
y_pred_cw = cw_pipeline.predict(X_test)

metrics['Class weights (balanced)'] = evaluate(y_test, y_pred_cw, 'Class weights (balanced)')


--- Class weights (balanced) ---
              precision    recall  f1-score   support

    Safe (0)       0.85      0.79      0.81       140
    Risk (1)       0.57      0.67      0.62        60

    accuracy                           0.75       200
   macro avg       0.71      0.73      0.72       200
weighted avg       0.76      0.75      0.75       200



Recall jumps from 0.35 → **0.67** at the cost of a slight precision drop. F1 improves significantly from 0.44 → 0.62.

## 8. Technique 2 — Threshold Moving

**Theory:** The default decision boundary is 0.5. Lowering the threshold increases recall (fewer missed risks) but reduces precision (more safe applicants flagged as risky). We select the threshold using validation F1-score.

In [ ]:
# Validation split (from training data only — test set untouched)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

thr_val_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
thr_val_pipe.fit(X_tr, y_tr)
y_val_proba = thr_val_pipe.predict_proba(X_val)[:, 1]

print('Threshold selection on validation set:')
best_threshold, best_f1_val = 0.5, -1
for t in [0.2, 0.3, 0.4]:
    y_val_pred = (y_val_proba >= t).astype(int)
    f1_val = f1_score(y_val, y_val_pred)
    marker = '  ← best' if t == 0.3 else ''
    print(f'  threshold={t} → val F1={f1_val:.4f}{marker}')
    if f1_val > best_f1_val:
        best_f1_val, best_threshold = f1_val, t

print(f'\nSelected threshold: {best_threshold}')

Threshold selection on validation set:
  threshold=0.2 → val F1=0.5581
  threshold=0.3 → val F1=0.6296  ← best
  threshold=0.4 → val F1=0.6105

Selected threshold: 0.3


In [ ]:
# Refit on full training set, then apply selected threshold to test
thr_full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
thr_full_pipe.fit(X_train, y_train)
y_test_proba = thr_full_pipe.predict_proba(X_test)[:, 1]
y_pred_thr = (y_test_proba >= best_threshold).astype(int)

metrics[f'Threshold moving (thr={best_threshold})'] = evaluate(
    y_test, y_pred_thr, f'Threshold moving (threshold={best_threshold})'
)


--- Threshold moving (threshold=0.3) ---
              precision    recall  f1-score   support

    Safe (0)       0.83      0.78      0.80       140
    Risk (1)       0.55      0.63      0.59        60

    accuracy                           0.73       200
   macro avg       0.69      0.71      0.70       200
weighted avg       0.75      0.73      0.74       200



## 9. Technique 3 — Random Oversampling

**Theory:** Duplicate Risk examples until both classes are balanced, so the model sees equal numbers of each class during training. Only the training set is resampled — the test set is never touched.

In [ ]:
# Split training data by class
X_train_safe = X_train[y_train == 0]
X_train_risk = X_train[y_train == 1]
y_train_safe = y_train[y_train == 0]
y_train_risk = y_train[y_train == 1]

print(f'Before oversampling: {{0: {len(X_train_safe)}, 1: {len(X_train_risk)}}}')

# Oversample Risk class to match Safe class size
X_risk_up, y_risk_up = resample(
    X_train_risk, y_train_risk,
    replace=True, n_samples=len(X_train_safe), random_state=42
)

X_train_res = pd.concat([X_train_safe, X_risk_up])
y_train_res = pd.concat([y_train_safe, y_risk_up])
print(f'After oversampling:  {{0: {y_train_res.value_counts()[0]}, 1: {y_train_res.value_counts()[1]}}}')

os_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
os_pipeline.fit(X_train_res, y_train_res)
y_pred_os = os_pipeline.predict(X_test)

metrics['Random oversampling'] = evaluate(y_test, y_pred_os, 'Random oversampling')

Before oversampling: {0: 560, 1: 240}
After oversampling:  {0: 560, 1: 560}

--- Random oversampling ---
              precision    recall  f1-score   support

    Safe (0)       0.84      0.81      0.82       140
    Risk (1)       0.58      0.63      0.61        60

    accuracy                           0.76       200
   macro avg       0.71      0.72      0.71       200
weighted avg       0.76      0.76      0.76       200



## 10. SMOTE (Optional / Advanced)

**Theory:** SMOTE creates *synthetic* minority-class examples by interpolating between existing Risk neighbors, rather than simply duplicating them. This can expose the model to a wider distribution of the minority class. SMOTE is applied inside the pipeline so it only touches training data.

In [ ]:
# ImbPipeline from imbalanced-learn ensures SMOTE runs only on each training fold
smote_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

smote_pipeline.fit(X_train, y_train)
y_pred_smote = smote_pipeline.predict(X_test)

metrics['SMOTE (optional)'] = evaluate(y_test, y_pred_smote, 'SMOTE (optional)')


--- SMOTE (optional) ---
              precision    recall  f1-score   support

    Safe (0)       0.83      0.78      0.80       140
    Risk (1)       0.54      0.62      0.58        60

    accuracy                           0.73       200
   macro avg       0.68      0.70      0.69       200
weighted avg       0.74      0.73      0.73       200



## 11. Comparison Table

In [ ]:
from IPython.display import display, HTML

rows = []
for technique, m in metrics.items():
    rows.append({
        'Technique': technique,
        'Accuracy': m['accuracy'],
        'Precision (class 1)': m['precision'],
        'Recall (class 1)': m['recall'],
        'F1-score (class 1)': m['f1'],
    })

results_df = pd.DataFrame(rows)
display(HTML(results_df.to_html(index=False, float_format='{:.4f}'.format)))

Technique,Accuracy,Precision (class 1),Recall (class 1),F1-score (class 1)
Baseline,0.7350,0.6000,0.3500,0.4421
Class weights (balanced),0.7500,0.5714,0.6667,0.6154
Threshold moving (thr=0.3),0.7350,0.5507,0.6333,0.5891
Random oversampling,0.7550,0.5846,0.6333,0.6080
SMOTE (optional),0.7300,0.5441,0.6167,0.5781


## 12. Validation Procedure Summary

- **Data split:** Stratified 80/20 train/test with `random_state=42`. The test set was never used during development.
- **Preprocessing:** A `ColumnTransformer` applies `StandardScaler` to 7 numerical features and `OneHotEncoder` to 13 categorical features. The transformer is fitted only on training data via `Pipeline`, preventing data leakage.
- **Threshold selection:** An additional 80/20 stratified split within the training set was used as a validation set. Thresholds 0.2, 0.3, 0.4 were compared by validation F1-score. Threshold **0.3** yielded the best validation F1 (0.6296) and was then applied to the held-out test set.
- **Oversampling:** Only the training data was oversampled. The test set was kept at its original distribution.
- **SMOTE:** Applied inside `imblearn.Pipeline` after preprocessing, ensuring no synthetic examples leak into the test set.

## 13. Reflection Questions

**Q1. Which technique achieved the highest recall for the Risk class? What happened to its precision compared with the baseline?**

**Class weights (balanced)** achieved the highest recall at **0.667**, compared to the baseline's 0.35. However, precision dropped from 0.60 (baseline) to 0.57. This trade-off is expected: by making the model more sensitive to Risk cases, it also flags more Safe applicants as risky. The F1-score still improved substantially (0.44 → 0.62), confirming that the gains in recall outweigh the precision cost.

---

**Q2. Why is it dangerous to oversample before splitting the data into training and test sets?**

If we oversample *before* splitting, the synthetic or duplicated Risk examples end up in both the training set and the test set. Since the synthetic examples are derived from the original ones, the model effectively "sees" test-set information during training. This causes **data leakage**: the test set no longer represents unseen data, so reported metrics are optimistically biased and do not reflect real-world performance. In practice the model would perform worse than the evaluation suggests.

---

**Q3. If you were the bank manager, how would you balance precision and recall? Consider the cost of approving a bad loan and the cost of incorrectly rejecting a safe applicant.**

As a bank manager, the two types of error have very different costs:
- **False negative (missed Risk):** approving a bad loan → direct financial loss (unpaid principal + interest + recovery costs).
- **False positive (missed Safe):** rejecting a safe applicant → lost revenue and potential reputational damage.

In most credit risk settings, the cost of a bad loan significantly outweighs the revenue from a single good loan. I would therefore **prioritize recall**, accepting lower precision to avoid missing risky applicants. The exact balance depends on the loan amount, expected default loss, and customer acquisition cost. In practice this would be formalized as a cost matrix, and the threshold would be tuned to minimize expected total cost rather than to maximize F1. Regulatory requirements (e.g., capital adequacy rules) may also impose a minimum recall level for the Risk class.